In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import mean_absolute_error, r2_score
from catboost import CatBoostRegressor

PARQUET_PATH = Path("/Users/zhasik/Desktop/krisha/data/index/index.parquet")
PROJECT_ROOT = Path("/Users/zhasik/Desktop/krisha")
PROC_DIR = PROJECT_ROOT / "data" / "processed"


In [2]:
df = pd.read_parquet(PARQUET_PATH)

CURRENT_YEAR = 2026
df_feat = df.copy()
df_feat["ad_id_str"] = df_feat["ad_id"].astype(str)

# new optional fields for v2 retrain (safe if columns are missing)
if "residential_complex" not in df_feat.columns:
    df_feat["residential_complex"] = "unknown"
df_feat["residential_complex"] = df_feat["residential_complex"].fillna("unknown").astype(str)
df_feat["residential_complex"] = df_feat["residential_complex"].replace({"": "unknown", "None": "unknown", "nan": "unknown"})

for c in ["latitude", "longitude"]:
    if c not in df_feat.columns:
        df_feat[c] = np.nan
    df_feat[c] = pd.to_numeric(df_feat[c], errors="coerce")

df_feat["has_geo"] = (df_feat["latitude"].notna() & df_feat["longitude"].notna()).astype(int)
df_feat["has_residential_complex"] = (df_feat["residential_complex"] != "unknown").astype(int)

if df_feat["latitude"].notna().any():
    df_feat["latitude"] = df_feat["latitude"].fillna(df_feat["latitude"].median())
else:
    df_feat["latitude"] = 43.238949

if df_feat["longitude"].notna().any():
    df_feat["longitude"] = df_feat["longitude"].fillna(df_feat["longitude"].median())
else:
    df_feat["longitude"] = 76.889709

df_feat["floor_ratio"] = (df_feat["floor"] / df_feat["floors_total"]).clip(0, 1)
df_feat["is_first"] = (df_feat["floor"] == 1).astype(int)
df_feat["is_last"]  = (df_feat["floor"] == df_feat["floors_total"]).astype(int)
df_feat["building_age"] = CURRENT_YEAR - df_feat["year_built"]

TARGET = "log_price_per_m2"

FEATURES_NUM = [
    "area","rooms","floor","floors_total","floor_ratio","is_first","is_last","building_age",
    "latitude","longitude","has_geo","has_residential_complex"
]
FEATURES_CAT = ["district","building_type","residential_complex"]


In [3]:
ad_ids = np.load(PROC_DIR / "clip_vitb32_ad_ids.npy")
ad_emb = np.load(PROC_DIR / "clip_vitb32_ad_emb.npy")

emb_dim = ad_emb.shape[1]
emb_cols = [f"clip_{i:03d}" for i in range(emb_dim)]
emb_df = pd.DataFrame(ad_emb, columns=emb_cols)
emb_df["ad_id_str"] = ad_ids.astype(str)

dfm = df_feat.merge(emb_df, on="ad_id_str", how="inner")
print("merged shape:", dfm.shape)
dfm.head(2)


merged shape: (1879, 536)


,ad_id,url,price,area,price_per_m2,log_price_per_m2,rooms,district,building_type,residential_complex,...,clip_502,clip_503,clip_504,clip_505,clip_506,clip_507,clip_508,clip_509,clip_510,clip_511
0,1008270130,https://krisha.kz/a/show/1008270130,33000000,40.0,825000.000000,13.623139,2,Медеуский р-н,монолитный,Mereke,...,-0.001001,-0.002158,0.027183,-0.004496,0.017282,0.022064,0.018169,0.028078,0.010147,0.03807
1,1002526087,https://krisha.kz/a/show/1002526087,56500000,75.1,752330.226365,13.530931,3,Алмалинский р-н,монолитный,Auezov City,...,0.007158,-0.017094,0.018038,-0.017241,0.021059,-0.007354,0.000488,-0.030866,-0.015792,-0.00501


In [4]:
train_ids = pd.read_csv(PROC_DIR / "train_ad_ids.csv", header=None).iloc[:,0].astype(str).tolist()
val_ids   = pd.read_csv(PROC_DIR / "val_ad_ids.csv", header=None).iloc[:,0].astype(str).tolist()
test_ids  = pd.read_csv(PROC_DIR / "test_ad_ids.csv", header=None).iloc[:,0].astype(str).tolist()

train_mask = dfm["ad_id_str"].isin(train_ids)
val_mask   = dfm["ad_id_str"].isin(val_ids)
test_mask  = dfm["ad_id_str"].isin(test_ids)

print(train_mask.sum(), val_mask.sum(), test_mask.sum())


1315 282 282


In [5]:
FEATURES = FEATURES_NUM + FEATURES_CAT + emb_cols

X_train = dfm.loc[train_mask, FEATURES].copy()
y_train = dfm.loc[train_mask, TARGET]

X_val = dfm.loc[val_mask, FEATURES].copy()
y_val = dfm.loc[val_mask, TARGET]

X_test = dfm.loc[test_mask, FEATURES].copy()
y_test = dfm.loc[test_mask, TARGET]

for c in FEATURES_CAT:
    X_train[c] = X_train[c].fillna("unknown").astype(str)
    X_val[c] = X_val[c].fillna("unknown").astype(str)
    X_test[c] = X_test[c].fillna("unknown").astype(str)

cat_features_idx = [X_train.columns.get_loc(c) for c in FEATURES_CAT]

model = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    early_stopping_rounds=200,
    verbose=200
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features_idx
)


0:	learn: 0.2938836	test: 0.3212038	best: 0.3212038 (0)	total: 220ms	remaining: 14m 38s
200:	learn: 0.1077239	test: 0.1956553	best: 0.1956553 (200)	total: 10.3s	remaining: 3m 14s
400:	learn: 0.0583729	test: 0.1831075	best: 0.1830920 (399)	total: 19.9s	remaining: 2m 58s
600:	learn: 0.0345409	test: 0.1809155	best: 0.1809072 (594)	total: 29.8s	remaining: 2m 48s
800:	learn: 0.0227327	test: 0.1803751	best: 0.1803611 (799)	total: 39.9s	remaining: 2m 39s
1000:	learn: 0.0149450	test: 0.1801149	best: 0.1800935 (983)	total: 50.1s	remaining: 2m 30s
1200:	learn: 0.0101366	test: 0.1800331	best: 0.1800252 (1143)	total: 1m	remaining: 2m 20s
1400:	learn: 0.0072677	test: 0.1800000	best: 0.1799950 (1305)	total: 1m 10s	remaining: 2m 10s
1600:	learn: 0.0053559	test: 0.1799828	best: 0.1799712 (1539)	total: 1m 20s	remaining: 2m 1s
1800:	learn: 0.0040263	test: 0.1799504	best: 0.1799306 (1712)	total: 1m 31s	remaining: 1m 51s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.1799306305
bestI

In [6]:
pred_log = model.predict(X_test)

r2 = r2_score(y_test.values, pred_log)
mae_ppm2 = mean_absolute_error(np.exp(y_test.values), np.exp(pred_log))

ppm2_pred = np.exp(pred_log)
price_pred = ppm2_pred * X_test["area"].astype(float).values
price_true = np.exp(y_test.values) * X_test["area"].astype(float).values
mae_price = mean_absolute_error(price_true, price_pred)

print("TEST R2(log):", round(r2, 4))
print("TEST MAE ppm2:", f"{mae_ppm2:,.0f} ₸/м²")
print("TEST MAE price:", f"{mae_price:,.0f} ₸")


TEST R2(log): 0.6452
TEST MAE ppm2: 104,363 ₸/м²
TEST MAE price: 8,824,144 ₸


In [7]:
OUT_DIR = PROJECT_ROOT / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

path = OUT_DIR / "catboost_tabular_plus_clip_vitb32.cbm"
model.save_model(str(path))
print("Saved:", path)


Saved: /Users/zhasik/Desktop/krisha/models/catboost_tabular_plus_clip_vitb32.cbm


In [8]:
import json
from pathlib import Path

PROJECT_ROOT = Path("/Users/zhasik/Desktop/krisha")
OUT = PROJECT_ROOT / "models"
OUT.mkdir(parents=True, exist_ok=True)

v2_meta = {
  "version": "v2_tabular_plus_clip_vitb32_mean7",
  "current_year": 2026,
  "target": "log_price_per_m2",
  "features_num": FEATURES_NUM,
  "features_cat": FEATURES_CAT,
  "clip": {
    "library": "open_clip",
    "model_name": "ViT-B-32",
    "pretrained": "laion2b_s34b_b79k",
    "aggregation": "mean",
    "max_images": 7,
    "normalize_per_image": True,
    "normalize_agg": True
  },
  "model_file": "catboost_tabular_plus_clip_vitb32.cbm"
}

(OUT / "v2_metadata.json").write_text(json.dumps(v2_meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", OUT / "v2_metadata.json")


saved: /Users/zhasik/Desktop/krisha/models/v2_metadata.json
